In [ ]:
import sys
import os
sys.path.append("..")
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from utils.retrieve_hpc_data import ensure_local_file

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Load data Temporal
orig_rel = "../wandb_logs/knn-mlp-regression-relative-multiseed-texture-intensity/evaluations/auo7f6xv/predictions.pt"
both_rel = "../wandb_logs/knn-mlp-regression-relative-multiseed-texture-intensity/evaluations/fhol3yv1/predictions.pt"
orig_local = ensure_local_file(orig_rel)
both_local = ensure_local_file(both_rel)
data_original = torch.load(orig_local, weights_only=False)
data_both = torch.load(both_local, weights_only=False)

data = {
    "original": data_original,
    "both_augmented": data_both,
}

for split in ["train", "val"]:
    n_dims = data_original[split]["Y"].shape[1]
    fig, axes = plt.subplots(1, n_dims, figsize=(6*n_dims, 5))
    axes = np.atleast_1d(axes)

    for d, ax in enumerate(axes):
        mse_text = []  # store MSEs for legend or annotation
        for key, val in data.items():
            Y = val[split]["Y"].cpu().numpy()
            preds = val[split]["preds"].cpu().numpy()
            mse = np.mean((Y[:, d] - preds[:, d])**2)
            # mse_text.append(f"{key}: {mse:.4f}")
            ax.scatter(Y[:, d], preds[:, d], alpha=0.4, label=f"{key} (MSE={mse:.4f})")

        # Diagonal reference line
        min_val, max_val = Y[:, d].min(), Y[:, d].max()
        ax.plot([min_val, max_val], [min_val, max_val], "r--")
        # ax.axis("equal")

        # Titles and labels
        ax.set_title(f"{split.upper()} dim {d}")
        ax.set_xlabel("True")
        ax.set_ylabel("Predicted")
        ax.legend()


    plt.tight_layout()
    plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np


data = {
    "original": data_original,
    "both_augmented": data_both,
}


# --- Error histograms ---
for split in ["train", "val"]:
    n_dims = data_original[split]["Y"].shape[1]
    fig, axes = plt.subplots(1, n_dims, figsize=(6*n_dims, 4))
    axes = np.atleast_1d(axes)

    for d, ax in enumerate(axes):
        for key, val in data.items():
            Y = val[split]["Y"].cpu().numpy()
            preds = val[split]["preds"].cpu().numpy()
            errors = preds[:, d] - Y[:, d]
            ax.hist(errors, bins=40, alpha=0.5, label=key, density=True)
        
        ax.set_title(f"{split.upper()} dim {d} — Error Distribution")
        ax.set_xlabel("Prediction Error (pred - true)")
        ax.set_ylabel("Density")
        ax.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
for split in ["train", "val"]:
    n_dims = data_original[split]["Y"].shape[1]
    fig, axes = plt.subplots(n_dims, 1, figsize=(10, 5*n_dims))
    axes = np.atleast_1d(axes)

    for d, ax in enumerate(axes):

        ax.plot(data_original[split]["data_array"]['t'][:],data_original[split]["preds"][:,0], label="original")
        ax.plot(data_both[split]["data_array"]['t'][:],data_both[split]["preds"][:,0], label="both_augmented")
        ax.plot(data_original[split]["data_array"]['t'][:],data_original[split]["Y"][:,0], color='black',label="GT")
            
        
        ax.set_title(f"{split.upper()} dim {d} — predictions over time")
        ax.set_xlabel("Time")
        ax.set_ylabel("Prediction")
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def compute_vector_errors(preds, Y):
    """
    Compute multiple vector prediction metrics.

    Args:
        preds: [N, D] predicted vectors
        Y: [N, D] ground truth vectors
    Returns:
        dict with metrics
    """
    import numpy as np

    errors = np.linalg.norm(preds - Y, axis=1)
    L1 = np.mean(np.abs(preds - Y))
    L1_GT = np.mean(np.abs(np.mean(Y, axis=0) - Y))
    L2 = np.mean((preds - Y)**2)
    L2_GT = np.mean((np.mean(Y, axis=0) - Y)**2)
    EPE = np.mean(errors)
    EPE_GT = np.mean(np.linalg.norm(np.mean(Y, axis=0) - Y, axis=1))
    acc_1px = np.mean(errors < 1.0)
    acc_2px = np.mean(errors < 2.0)
    acc_3px = np.mean(errors < 3.0)

    dot = np.sum(preds * Y, axis=1)
    norm_pred = np.linalg.norm(preds, axis=1)
    norm_true = np.linalg.norm(Y, axis=1)
    eps = 1e-8
    cosine = dot / (norm_pred * norm_true + eps)
    cosine = np.clip(cosine, -1.0, 1.0)
    angular_error = np.degrees(np.arccos(cosine))
    mean_angle = np.mean(angular_error)
    
    Y_mean = np.mean(Y, axis=0, keepdims=True).repeat(Y.shape[0], axis=0)
    dot = np.sum(Y_mean * Y, axis=1)
    norm_Y_mean = np.linalg.norm(Y_mean, axis=1)
    norm_true = np.linalg.norm(Y, axis=1)
    eps = 1e-8
    cosine = dot / (norm_Y_mean * norm_true + eps)
    cosine = np.clip(cosine, -1.0, 1.0)
    angular_error = np.degrees(np.arccos(cosine))
    mean_angle_GT = np.mean(angular_error)
    

    return {
        "L1": L1,
        "L1_GT": L1_GT,
        "L2": L2,
        "L2_GT": L2_GT,
        "EPE": EPE,
        "EPE_GT": EPE_GT,
        "1px_acc": acc_1px,
        "2px_acc": acc_2px,
        "3px_acc": acc_3px,
        "angular_error_deg": mean_angle,
        "angular_error_deg_GT": mean_angle_GT
    }
    
for key, val in data.items():   
    Y = val["val"]["Y"].cpu().numpy()
    preds = val["val"]["preds"].cpu().numpy()
    metrics = compute_vector_errors(preds, Y)
    print(f"Metrics for {key}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")


In [ ]:
filter_values.min()

In [ ]:
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Choose split and model
split = "val"
model_key = "both_augmented"  # or "original"

# Load arrays
# NOTE: data_original/data_both are prepared earlier; ensure_local_file handled copying if needed
Y = data[model_key][split]["Y"].cpu().numpy()
preds = data[model_key][split]["preds"].cpu().numpy()
x = data[model_key][split]["data_array"]["x"]
y = data[model_key][split]["data_array"]["y"]
eig1 = data[model_key][split]["X"][:, 2].cpu().numpy()  # assuming eig1 is stored here
eig2 = data[model_key][split]["X"][:, 3].cpu().numpy()  # assuming eig2 is stored here
filter_values = data[model_key][split]["X"][:, 4].cpu().numpy()  # assuming filter values are stored here

harris_k = 0.04

harris_corner_score = eig1 * eig2 - harris_k * (eig1 + eig2)**2

# --- Compute endpoint error (Euclidean distance)
errors = np.linalg.norm(preds - Y, axis=1)

# Animation parameters
window_size = 1000
step_size = 200       # controls how much we move per frame
fps = 10

ploting_variable = filter_values  # Change this to eig1, eig2, harris_corner_score, or filter_values
vmin = np.percentile(ploting_variable, 5)  # 5 percentile of the chosen variable
vmax = np.percentile(ploting_variable, 95)  # 95 percentile of the chosen variable
# --- Create figure
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter([], [], c=[], cmap='plasma', vmin=vmin, vmax=vmax, s=5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Spatial Error Animation ({model_key}, {split})")

# Precompute limits
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Endpoint Error (EPE)")

# --- Update function
def update(frame):
    start = frame * step_size
    end = start + window_size
    if end > len(ploting_variable):
        end = len(ploting_variable)
    sc.set_offsets(np.c_[x[start:end], y[start:end]])
    sc.set_array(ploting_variable[start:end])
    ax.set_title(f"Samples {start}-{end} | {model_key} ({split})")
    return sc,

# Number of frames
frames = (len(ploting_variable) - window_size) // step_size

ani = FuncAnimation(fig, update, frames=frames, interval=1000/fps, blit=True)

plt.show()
HTML(ani.to_jshtml())

In [ ]:
# compute filter_values histogram
filter_values = data[model_key][split]["X"][:, 4].cpu().numpy()  # assuming filter values are stored here
plt.figure(figsize=(8,6))
plt.hist(filter_values, bins=50, color='gray', alpha=0.7)
plt.xlabel("Filter Values")
plt.ylabel("Frequency")
plt.title(f"Filter Values Histogram ({model_key}, {split})")
plt.show()

In [ ]:
original_error = np.linalg.norm(data["original"]["val"]["preds"].cpu().numpy() - data["original"]["val"]["Y"].cpu().numpy(), axis=1)
both_error = np.linalg.norm(data["both_augmented"]["val"]["preds"].cpu().numpy() - data["both_augmented"]["val"]["Y"].cpu().numpy(), axis=1)
# scatter plot error vs harris corner score
plt.figure(figsize=(8,6))
plt.scatter(data["original"]["val"]["X"][:, 4].cpu().numpy(), original_error, alpha=0.5, s=5)
plt.xlabel("Harris Corner Score")       
plt.ylabel("Endpoint Error (EPE)")
plt.title(f"Error vs filter values ({model_key}, {split})")
plt.scatter(data["both_augmented"]["val"]["X"][:, 4].cpu().numpy(), both_error, alpha=0.5, s=5, color='orange')
plt.legend(["original", "both_augmented"])
plt.show()


In [ ]:
# Choose split and model
split = "val"
model_key = "original"

# Load arrays
# NOTE: data_original/data_both are prepared earlier; ensure_local_file handled copying if needed
Y = data[model_key][split]["Y"].cpu().numpy()
preds = data[model_key][split]["preds"].cpu().numpy()
x_original = data[model_key][split]["data_array"]["x"]
y_original = data[model_key][split]["data_array"]["y"]
t_original = data[model_key][split]["data_array"]["t"]
eig1_original = data[model_key][split]["X"][:, 2].cpu().numpy()  # assuming eig1 is stored here
eig2_original = data[model_key][split]["X"][:, 3].cpu().numpy()  # assuming eig2 is stored here
filter_values_original = data[model_key][split]["X"][:, 4].cpu().numpy()  # assuming filter values are stored here
# --- Compute endpoint error (Euclidean distance)
errors_original = np.linalg.norm(preds - Y, axis=1)
errors_both = np.linalg.norm(preds - Y, axis=1)
dot = np.sum(preds * Y, axis=1)
norm_pred = np.linalg.norm(preds, axis=1)
norm_true = np.linalg.norm(Y, axis=1)
eps = 1e-8
cosine = dot / (norm_pred * norm_true + eps)
cosine = np.clip(cosine, -1.0, 1.0)
angular_error_original = np.degrees(np.arccos(cosine))
print(f"number of events: {x_original.shape[0]}")
print(f"min x_original: {x_original.min()}, max x_original: {x_original.max()}")
print(f"min y_original: {y_original.min()}, max y_original: {y_original.max()}")
print(f"min t_original: {t_original.min()}, max t_original: {t_original.max()}")
print(f"min eig1_original: {eig1_original.min()}, max eig1_original: {eig1_original.max()}")
print(f"min eig2_original: {eig2_original.min()}, max eig2_original: {eig2_original.max()}")
print(f"min filter_values_original: {filter_values_original.min()}, max filter_values_original: {filter_values_original.max()}")

print("---"*20)
model_key = "both_augmented"

# Load arrays
# NOTE: data_original/data_both are prepared earlier; ensure_local_file handled copying if needed
Y = data[model_key][split]["Y"].cpu().numpy()
preds = data[model_key][split]["preds"].cpu().numpy()
x_both = data[model_key][split]["data_array"]["x"]
y_both = data[model_key][split]["data_array"]["y"]
t_both = data[model_key][split]["data_array"]["t"]
eig1_both = data[model_key][split]["X"][:, 2].cpu().numpy()  # assuming eig1 is stored here
eig2_both = data[model_key][split]["X"][:, 3].cpu().numpy()  # assuming eig2 is stored here
filter_values_both = data[model_key][split]["X"][:, 4].cpu().numpy()  # assuming filter values are stored
print(f"number of events: {x_both.shape[0]}")
print(f"min x_both: {x_both.min()}, max x_both: {x_both.max()}")
print(f"min y_both: {y_both.min()}, max y_both: {y_both.max()}")
print(f"min t_both: {t_both.min()}, max t_both: {t_both.max()}")
print(f"min eig1_both: {eig1_both.min()}, max eig1_both: {eig1_both.max()}")
print(f"min eig2_both: {eig2_both.min()}, max eig2_both: {eig2_both.max()}")
print(f"min filter_values_both: {filter_values_both.min()}, max filter_values_both: {filter_values_both.max()}")
# --- Compute endpoint error (Euclidean distance)
errors_both = np.linalg.norm(preds - Y, axis=1)
dot = np.sum(preds * Y, axis=1)
norm_pred = np.linalg.norm(preds, axis=1)
norm_true = np.linalg.norm(Y, axis=1)
eps = 1e-8
cosine = dot / (norm_pred * norm_true + eps)
cosine = np.clip(cosine, -1.0, 1.0)
angular_error_both = np.degrees(np.arccos(cosine))

x = np.concatenate([x_original, x_both + 256])
y = np.concatenate([y_original, y_both])
t = np.concatenate([t_original, t_both])
eig1 = np.concatenate([eig1_original, eig1_both])
eig2 = np.concatenate([eig2_original, eig2_both])
filter_values = np.concatenate([filter_values_original, filter_values_both])
errors = np.concatenate([errors_original, errors_both])
angular_error = np.concatenate([angular_error_original, angular_error_both])

# sort by time
sort_idx = np.argsort(t)
x = x[sort_idx]
y = y[sort_idx]
t = t[sort_idx]
eig1 = eig1[sort_idx]
eig2 = eig2[sort_idx]
filter_values = filter_values[sort_idx]
errors = errors[sort_idx]
angular_error = angular_error[sort_idx]

In [ ]:

# Animation parameters
window_size = 1000
step_size = 200        # controls how much we move per frame
fps = 10

ploting_variable = eig2  # Change this to eig1, eig2, or filter_values
vmin = np.percentile(ploting_variable, 5)  # 5 percentile of
vmax = np.percentile(ploting_variable, 95)  # 95 percentile of the chosen variable

# --- Create figure
fig, ax = plt.subplots(figsize=(12, 6))
sc = ax.scatter([], [], c=[], cmap='plasma', vmin=vmin, vmax=vmax, s=5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Spatial Error Animation (left original right both_augmented, {split})")

# Precompute limits
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Endpoint Error (EPE)")

# --- Update function
def update(frame):
    start = frame * step_size
    end = start + window_size
    if end > len(ploting_variable):
        end = len(ploting_variable)
    sc.set_offsets(np.c_[x[start:end], y[start:end]])
    sc.set_array(ploting_variable[start:end])
    ax.set_title(f"Spatial Error Animation (left original right both_augmented, {split})")
    return sc,

# Number of frames
frames = (len(ploting_variable) - window_size) // step_size

ani = FuncAnimation(fig, update, frames=frames, interval=1000/fps, blit=True)

plt.show()
# ani.save(f"error_animation_{model_key}_{split}.gif", writer="pillow", fps=fps)

HTML(ani.to_jshtml())

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Choose split and model
split = "val"
model_key = "both_augmented"  # or "original"

# Load arrays
# NOTE: data_original/data_both are prepared earlier; ensure_local_file handled copying if needed
Y = data[model_key][split]["Y"].cpu().numpy()
preds = data[model_key][split]["preds"].cpu().numpy()
x = data[model_key][split]["data_array"]["x"]
y = data[model_key][split]["data_array"]["y"]

# --- Compute angular error (in degrees)
dot = np.sum(preds * Y, axis=1)
norm_pred = np.linalg.norm(preds, axis=1)
norm_true = np.linalg.norm(Y, axis=1)
eps = 1e-8
cosine = dot / (norm_pred * norm_true + eps)
cosine = np.clip(cosine, -1.0, 1.0)
angular_error = np.degrees(np.arccos(cosine))

# Animation parameters
window_size = 1000
step_size = 200
fps = 10

# --- Create figure
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter([], [], c=[], cmap='turbo', vmin=0, vmax=np.percentile(angular_error, 95), s=5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Angular Error Animation ({model_key}, {split})")

# Precompute limits
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Angular Error (°)")

# --- Update function
def update(frame):
    start = frame * step_size
    end = start + window_size
    if end > len(angular_error):
        end = len(angular_error)
    sc.set_offsets(np.c_[x[start:end], y[start:end]])
    sc.set_array(angular_error[start:end])
    ax.set_title(f"Samples {start}-{end} | {model_key} ({split})")
    return sc,

# Number of frames
frames = (len(angular_error) - window_size) // step_size

ani = FuncAnimation(fig, update, frames=frames, interval=1000/fps, blit=True)

plt.show()
ani.save(f"angular_error_animation_{model_key}_{split}.gif", writer="pillow", fps=fps)
HTML(ani.to_jshtml())

In [ ]:
# Choose split and model
split = "val"
model_key = "original"  # or "original"

# Load arrays
# NOTE: data_original/data_both are prepared earlier; ensure_local_file handled copying if needed
Y = data[model_key][split]["Y"].cpu().numpy()
preds = data[model_key][split]["preds"].cpu().numpy()
x = data[model_key][split]["data_array"]["x"]
y = data[model_key][split]["data_array"]["y"]

# --- Compute angular error (in degrees)
dot = np.sum(preds * Y, axis=1)
norm_pred = np.linalg.norm(preds, axis=1)
norm_true = np.linalg.norm(Y, axis=1)
eps = 1e-8
cosine = dot / (norm_pred * norm_true + eps)
cosine = np.clip(cosine, -1.0, 1.0)
angular_error = np.degrees(np.arccos(cosine))

# Animation parameters
window_size = 1000
step_size = 200
fps = 10

# --- Create figure
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter([], [], c=[], cmap='turbo', vmin=0, vmax=np.percentile(angular_error, 95), s=5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Angular Error Animation ({model_key}, {split})")

# Precompute limits
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Angular Error (°)")

# --- Update function
def update(frame):
    start = frame * step_size
    end = start + window_size
    if end > len(angular_error):
        end = len(angular_error)
    sc.set_offsets(np.c_[x[start:end], y[start:end]])
    sc.set_array(angular_error[start:end])
    ax.set_title(f"Samples {start}-{end} | {model_key} ({split})")
    return sc,

# Number of frames
frames = (len(angular_error) - window_size) // step_size

ani = FuncAnimation(fig, update, frames=frames, interval=1000/fps, blit=True)

plt.show()
ani.save(f"angular_error_animation_{model_key}_{split}.gif", writer="pillow", fps=fps)
HTML(ani.to_jshtml())

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Choose split and model
split = "val"
model_key = "original"

# Load arrays
Y = data[model_key][split]["Y"].cpu().numpy()
preds = data[model_key][split]["preds"].cpu().numpy()
x = data[model_key][split]["data_array"]["x"]
y = data[model_key][split]["data_array"]["y"]

# --- Compute angular error (in degrees)
dot = np.sum(preds * Y, axis=1)
norm_pred = np.linalg.norm(preds, axis=1)
norm_true = np.linalg.norm(Y, axis=1)
eps = 1e-8
cosine = dot / (norm_pred * norm_true + eps)
cosine = np.clip(cosine, -1.0, 1.0)
angular_error = np.degrees(np.arccos(cosine))

# Animation parameters
window_size = 1000
step_size = 200
fps = 10

# --- Create figure
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter([], [], c=[], cmap='turbo', vmin=0, vmax=np.percentile(angular_error, 95), s=5)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Angular Error Animation ({model_key}, {split})")

# Precompute limits
ax.set_xlim(x.min(), x.max())
ax.set_ylim(y.min(), y.max())
cb = plt.colorbar(sc, ax=ax)
cb.set_label("Angular Error (°)")

# --- Update function
def update(frame):
    start = frame * step_size
    end = start + window_size
    if end > len(angular_error):
        end = len(angular_error)
    sc.set_offsets(np.c_[x[start:end], y[start:end]])
    sc.set_array(angular_error[start:end])
    ax.set_title(f"Samples {start}-{end} | {model_key} ({split})")
    return sc,

# Number of frames
frames = (len(angular_error) - window_size) // step_size

ani = FuncAnimation(fig, update, frames=frames, interval=1000/fps, blit=True)

plt.show()
ani.save(f"angular_error_animation_{model_key}_{split}.gif", writer="pillow", fps=fps)
HTML(ani.to_jshtml())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_quiver(X, Y, title):
    plt.figure(figsize=(6,6))
    plt.quiver(X[:,0], X[:,1], Y[:,0], Y[:,1], angles="xy", scale_units="xy", scale=1, alpha=0.5)
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.axis("equal")
    plt.show()

# Example on test
X = data["val"]["X"].numpy()[:, :2]   # assuming first two cols are coordinates
Y_true = data["val"]["Y"].numpy()
Y_pred = data["val"]["preds"].numpy()

idx_range = np.arange(2000,2100)
plot_quiver(X[idx_range,:], Y_true[idx_range,:], "Ground Truth velocity")
plot_quiver(X[idx_range,:], Y_pred[idx_range,:], "Predicted velocity")
